In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GraphSAGE
from sklearn.linear_model import LogisticRegression
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Load the datasets
cora_dataset = Planetoid(root='/tmp/Citeseer', name='Citeseer')
data = cora_dataset[0]
data = data.to(device, 'x', 'edge_index')

train_loader = LinkNeighborLoader(
    data,
    batch_size=256,
    shuffle=True,
    neg_sampling_ratio=1.0,
    num_neighbors=[10, 10],
)

model = GraphSAGE(
    data.num_node_features,
    hidden_channels=64,
    num_layers=2,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


In [3]:
def train():
    model.train()

    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        h = model(batch.x, batch.edge_index)
        h_src = h[batch.edge_label_index[0]]
        h_dst = h[batch.edge_label_index[1]]
        pred = (h_src * h_dst).sum(dim=-1)
        loss = F.binary_cross_entropy_with_logits(pred, batch.edge_label)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.size(0)

    return total_loss / data.num_nodes

In [4]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index).cpu()

    clf = LogisticRegression()
    clf.fit(out[data.train_mask], data.y[data.train_mask])

    val_acc = clf.score(out[data.val_mask], data.y[data.val_mask])
    test_acc = clf.score(out[data.test_mask], data.y[data.test_mask])

    return val_acc, test_acc

In [5]:
for epoch in range(0, 200):
    loss = train()
    acc = test()[1]
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 3.5709, Accuracy: 0.3940
Epoch: 001, Loss: 2.8660, Accuracy: 0.4090
Epoch: 002, Loss: 2.8046, Accuracy: 0.3980
Epoch: 003, Loss: 2.7519, Accuracy: 0.3950
Epoch: 004, Loss: 2.6671, Accuracy: 0.4040
Epoch: 005, Loss: 2.6616, Accuracy: 0.4160
Epoch: 006, Loss: 2.6526, Accuracy: 0.4280
Epoch: 007, Loss: 2.6065, Accuracy: 0.4220
Epoch: 008, Loss: 2.6369, Accuracy: 0.4350
Epoch: 009, Loss: 2.5933, Accuracy: 0.3970
Epoch: 010, Loss: 2.6305, Accuracy: 0.3720
Epoch: 011, Loss: 2.6026, Accuracy: 0.4110
Epoch: 012, Loss: 2.5988, Accuracy: 0.4250
Epoch: 013, Loss: 2.6277, Accuracy: 0.4170
Epoch: 014, Loss: 2.5774, Accuracy: 0.4300
Epoch: 015, Loss: 2.6078, Accuracy: 0.4330
Epoch: 016, Loss: 2.5756, Accuracy: 0.4400
Epoch: 017, Loss: 2.5984, Accuracy: 0.4360
Epoch: 018, Loss: 2.6320, Accuracy: 0.4100
Epoch: 019, Loss: 2.6067, Accuracy: 0.4000
Epoch: 020, Loss: 2.5751, Accuracy: 0.3900
Epoch: 021, Loss: 2.5774, Accuracy: 0.3860
Epoch: 022, Loss: 2.5901, Accuracy: 0.3820
Epoch: 023,

In [6]:
torch.save(model.state_dict(), 'citeseer_gsage.pt')